In [268]:
import pandas as pd
import matplotlib.pyplot as plt

In [269]:
train = pd.read_csv("train_final.csv")
test = pd.read_csv("test_final.csv")
ids = test["row_id"]

In [270]:
train = train.sort_values("date", ascending=True)

In [271]:
train

,row_id,date,store_id,city,region,product_id,product_name,category,price,loyalty_day,is_holiday,holiday_name,demand
420500,0,2023-05-15,CM-BAC-01,Bacău,Moldova,P0001,Apă plată 2L,Beverages,4.51,0,0,NaN,3
420501,726,2023-05-15,CM-BAC-01,Bacău,Moldova,P0002,Apă minerală carbogazoasă 1.5L,Beverages,9.06,0,0,NaN,6
420502,1452,2023-05-15,CM-BAC-01,Bacău,Moldova,P0003,Cola 2L,Beverages,10.30,0,0,NaN,0
420503,2178,2023-05-15,CM-BAC-01,Bacău,Moldova,P0004,Suc natural portocale 1L,Beverages,8.65,0,0,NaN,1
420504,2904,2023-05-15,CM-BAC-01,Bacău,Moldova,P0005,Bere lager 0.5L,Beverages,4.51,0,0,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
575,418175,2025-05-10,CM-TIM-01,Timișoara,Banat,P0054,Mălai 1kg,Pantry,14.63,1,0,NaN,1
576,418901,2025-05-10,CM-TIM-01,Timișoara,Banat,P0055,Joc puzzle mini,Toys & Games,10.73,1,0,NaN,0
577,419627,2025-05-10,CM-TIM-01,Timișoara,Banat,P0056,Bandă elastică fitness,Sports & Hobbies,55.49,1,0,NaN,0
578,420353,2025-05-10,CM-TIM-01,Timișoara,Banat,P0057,Lingură silicon 30cm,Home & Kitchen,9.86,1,0,NaN,2


In [272]:
import pandas as pd
import numpy as np

def prep(train, test):

    train = train.copy()
    test = test.copy()

    # --------------------------
    # DATE FEATURES
    # --------------------------
    for df in [train, test]:
        df["date"] = pd.to_datetime(df["date"])

        df["day"] = df["date"].dt.day
        df["month"] = df["date"].dt.month
        df["year"] = df["date"].dt.year
        df["dayofweek"] = df["date"].dt.dayofweek
        df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)

        df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

        start_date = pd.Timestamp("2023-01-01")
        df["days_since_2023"] = (df["date"] - start_date).dt.days

    # --------------------------
    # TARGET ENCODING FEATURES (FROM TRAIN ONLY)
    # --------------------------

    global_mean = train["demand"].mean()

    # product_id mean
    group_cols = [
        "product_id",
        "region",
        "city",
        "category",
        "store_id"
    ]
    
    means = {
        col: train.groupby(col)["demand"].mean()
        for col in group_cols
    }
    for col in group_cols:
        train[f"{col}_mean"] = train[col].map(means[col])
        test[f"{col}_mean"] = test[col].map(means[col])

        # fill missing (important for test unseen categories)
        global_mean = train["demand"].mean()

        train[f"{col}_mean"] = train[f"{col}_mean"].fillna(global_mean)
        test[f"{col}_mean"] = test[f"{col}_mean"].fillna(global_mean)


    # --------------------------
    # DROP UNUSED COLUMNS
    # --------------------------
    train = train.drop(columns=["date", "row_id"])
    test = test.drop(columns=["date", "row_id"])

    # --------------------------
    # ONE-HOT ENCODING
    # --------------------------
    full = pd.concat([train, test], axis=0)
    full = pd.get_dummies(full)

    train = full.iloc[:len(train)].copy()
    test = full.iloc[len(train):].copy()

    return train, test

In [273]:
train, test = prep(train, test)

In [274]:
X_train = train.drop(columns="demand")
y_train = train["demand"]
X_test = test.drop(columns="demand")

In [275]:
train

,price,loyalty_day,is_holiday,demand,day,month,year,dayofweek,weekofyear,is_weekend,...,holiday_name_Sf. Andrei,holiday_name_Sf. Maria,holiday_name_Sf. Nicolae,holiday_name_Sf. Valentin,holiday_name_Unirea Principatelor,holiday_name_Vinerea Mare,holiday_name_Ziua Copilului,holiday_name_Ziua Femeii,holiday_name_Ziua Muncii,holiday_name_Ziua Națională
420500,4.51,0,0,3.0,15,5,2023,0,20,0,...,False,False,False,False,False,False,False,False,False,False
420501,9.06,0,0,6.0,15,5,2023,0,20,0,...,False,False,False,False,False,False,False,False,False,False
420502,10.30,0,0,0.0,15,5,2023,0,20,0,...,False,False,False,False,False,False,False,False,False,False
420503,8.65,0,0,1.0,15,5,2023,0,20,0,...,False,False,False,False,False,False,False,False,False,False
420504,4.51,0,0,0.0,15,5,2023,0,20,0,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
575,14.63,1,0,1.0,10,5,2025,5,19,1,...,False,False,False,False,False,False,False,False,False,False
576,10.73,1,0,0.0,10,5,2025,5,19,1,...,False,False,False,False,False,False,False,False,False,False
577,55.49,1,0,0.0,10,5,2025,5,19,1,...,False,False,False,False,False,False,False,False,False,False
578,9.86,1,0,2.0,10,5,2025,5,19,1,...,False,False,False,False,False,False,False,False,False,False


In [276]:
from catboost import CatBoostRegressor
from sklearn.svm import SVR

model = CatBoostRegressor(
    depth=8,
    learning_rate=0.03,
)

In [277]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, shuffle=False, test_size=0.2)

In [278]:
X_val

,price,loyalty_day,is_holiday,day,month,year,dayofweek,weekofyear,is_weekend,days_since_2023,...,holiday_name_Sf. Andrei,holiday_name_Sf. Maria,holiday_name_Sf. Nicolae,holiday_name_Sf. Valentin,holiday_name_Unirea Principatelor,holiday_name_Vinerea Mare,holiday_name_Ziua Copilului,holiday_name_Ziua Femeii,holiday_name_Ziua Muncii,holiday_name_Ziua Națională
84564,4.38,1,0,16,12,2024,0,51,0,715,...,False,False,False,False,False,False,False,False,False,False
84565,8.93,1,0,16,12,2024,0,51,0,715,...,False,False,False,False,False,False,False,False,False,False
84566,10.01,1,0,16,12,2024,0,51,0,715,...,False,False,False,False,False,False,False,False,False,False
84567,8.84,1,0,16,12,2024,0,51,0,715,...,False,False,False,False,False,False,False,False,False,False
84568,4.38,1,0,16,12,2024,0,51,0,715,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
575,14.63,1,0,10,5,2025,5,19,1,860,...,False,False,False,False,False,False,False,False,False,False
576,10.73,1,0,10,5,2025,5,19,1,860,...,False,False,False,False,False,False,False,False,False,False
577,55.49,1,0,10,5,2025,5,19,1,860,...,False,False,False,False,False,False,False,False,False,False
578,9.86,1,0,10,5,2025,5,19,1,860,...,False,False,False,False,False,False,False,False,False,False


In [279]:
model.fit(X_train, y_train, eval_set=(X_val, y_val))

0:	learn: 2.7800127	test: 2.5600051	best: 2.5600051 (0)	total: 21.5ms	remaining: 21.5s
1:	learn: 2.7531162	test: 2.5414287	best: 2.5414287 (1)	total: 37.8ms	remaining: 18.9s
2:	learn: 2.7278906	test: 2.5280335	best: 2.5280335 (2)	total: 54.7ms	remaining: 18.2s
3:	learn: 2.7029386	test: 2.5132936	best: 2.5132936 (3)	total: 71.8ms	remaining: 17.9s
4:	learn: 2.6792445	test: 2.4965094	best: 2.4965094 (4)	total: 89ms	remaining: 17.7s
5:	learn: 2.6578360	test: 2.4803248	best: 2.4803248 (5)	total: 105ms	remaining: 17.3s
6:	learn: 2.6375394	test: 2.4659209	best: 2.4659209 (6)	total: 120ms	remaining: 17s
7:	learn: 2.6172269	test: 2.4524063	best: 2.4524063 (7)	total: 137ms	remaining: 17s
8:	learn: 2.5983722	test: 2.4383625	best: 2.4383625 (8)	total: 151ms	remaining: 16.7s
9:	learn: 2.5798364	test: 2.4272178	best: 2.4272178 (9)	total: 168ms	remaining: 16.6s
10:	learn: 2.5620779	test: 2.4192588	best: 2.4192588 (10)	total: 184ms	remaining: 16.6s
11:	learn: 2.5443954	test: 2.4151268	best: 2.4151268 

CatBoostRegressor(depth=8, learning_rate=0.03, loss_function='RMSE')

In [280]:
pred = model.predict(X_test)

In [281]:
f = open("ans2.csv", 'w')

f.write("row_id,demand\n")

for i, p in enumerate(pred):
    f.write(str(ids[i]) + "," + str(p) + '\n')
f.close()